# Notebook 06: Claim Extraction, Independent Verification & Hallucination Detection
## Hallucination-Resistant Medical Report Generation Using Generative AI

### Central Research Focus:
1. Extract discrete atomic medical statements from generated reports.
2. Preserve polarity and clinical negation (e.g. 'No pleural effusion' vs 'Pleural effusion').
3. Perform **independent claim-level visual verification** using a separate visual classifier without circular self-verification.
4. Assign scientific verification status: `SUPPORTED`, `UNSUPPORTED`, `UNCERTAIN`, `CONTRADICTED`.
5. Compute uncertainty estimates and execute calibrated selective prediction/abstention.


In [ ]:
# 1. Module Imports
import sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/hallucination_medical_ai')]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.config import get_config
from src.verification.claim_extractor import MedicalClaimExtractor
from src.verification.verifier import IndependentHallucinationVerifier
from src.verification.uncertainty import UncertaintyEstimator

cfg = get_config()
print('Verification modules imported successfully.')


### 2. Extract Atomic Medical Claims from Report Text


In [ ]:
sample_report_text = (
    "The heart is mildly enlarged. There is no pleural effusion or pneumothorax. "
    "A bibasilar lung opacity is noted, suggestive of atelectasis. There is evidence of pulmonary edema."
)

extractor = MedicalClaimExtractor()
claims = extractor.extract_claims(sample_report_text)
print(f'Extracted {len(claims)} discrete claims:')
for c in claims:
    print(f'[{c.claim_id}] Finding: {c.normalized_finding:20s} | Polarity: {c.polarity:8s} | Text: "{c.original_text}"')


### 3. Independent Visual Evidence Verification
Compare extracted claims against predictions from an independent visual classifier.


In [ ]:
# Independent visual model predictions (simulated ground truth from independent classifier)
independent_visual_probs = {
    'Cardiomegaly': 0.82,
    'Pleural Effusion': 0.05,
    'Pneumothorax': 0.02,
    'Atelectasis': 0.70,
    'Lung Opacity': 0.65,
    'Edema': 0.12 # Notice report claims edema, but visual probability is low -> CONTRADICTION or UNSUPPORTED
}

verifier = IndependentHallucinationVerifier(cfg)
verification_results = verifier.verify_claims(claims, independent_visual_probs)

print('='*70)
print('INDEPENDENT VERIFICATION RESULTS')
print('='*70)
for res in verification_results:
    print(f'Claim {res.claim_id:4s} | Status: {res.status:13s} | Conf: {res.confidence:.2f} | Reason: {res.explanation}')


### 4. Calibrated Uncertainty Estimation and Selective Prediction
Estimate Expected Calibration Error (ECE) and execute selective abstention on ambiguous findings.


In [ ]:
uncertainty_estimator = UncertaintyEstimator(cfg)
calibrated_results = uncertainty_estimator.evaluate_uncertainty(verification_results)
print(f'Total Claims: {len(calibrated_results)}')
print(f'Supported: {sum(1 for r in calibrated_results if r.status == "SUPPORTED")}')
print(f'Unsupported: {sum(1 for r in calibrated_results if r.status == "UNSUPPORTED")}')
print(f'Contradicted: {sum(1 for r in calibrated_results if r.status == "CONTRADICTED")}')
print(f'Uncertain: {sum(1 for r in calibrated_results if r.status == "UNCERTAIN")}')
